# 토픽 분석
- 문서 속에 숨겨진 주제(Topic)를 자동으로 찾아내는 기법
- LDA는 다음과 같은 두 가지 기본 가정을 따른다.
    1. 문서는 여러 주제(토픽)의 조합이다.
        - 예: "스포츠 기사"는 50% 축구 + 30% 농구 + 20% 야구 주제로 구성될 수 있다.
    2. 각 주제는 여러 단어의 조합이다.
        - 예: 축구라는 주제는 "골", "선수", "리그", "경기" 등의 단어가 자주 등장한다.

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('data/배달의민족댓글.csv')
df = df.dropna()
df = df.reset_index()
df = df.drop(columns=['index','Unnamed: 0'])
df

,댓글
0,80분 걸린다길래 주문취소 하려고 주문내역에 들어가면 계속 최신 정보를 불러오지 못...
1,음식 하나 시키는데 우리나라 앱들은 1) 국내 번호 필요함. 번호 인증 필수 2) ...
2,왜이렇게 업데이트 할때마다 사용하기 점점 불편하게 바뀌는지? 클릭한번 더 해야되고 ...
3,"배달의 민족앱자체는 만족하나, 식사후 맛 리뷰 평점 자체는 클린하게 이뤄지진 못하는..."
4,장바구니가 너무 불편합니다. 비마트에서 여러가지를 담고 스크롤 올리고 내릴때 살짝 ...
...,...
455,우선 배달업체 광고가 너무 많습니다. 두번째는 주문직전 주소바꾸기 안되는게 매우매우...
456,배민 쭉 써왔고 쓴소리 하나 하려합니다. 중간다리 플랫폼으로서 식당/유저 사이 중재...
457,업데이트된거 디자인 너무 불편해요. 배민1 부분에서 음식점 둘러보는데 빨리 한거번에...
458,첫주문도 아닌데 첫주문 할인받으로 광고 계속오고 친구초대 하려하니까 주문내역이 없다...


In [6]:
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-21.0.11"

from konlpy.tag import Okt
import re
from tqdm import tqdm   # 진행 상황을 시각적으로 보여주는 라이브러리

In [7]:
def tokenize_text(text):
    text = re.sub(r'[^ㄱ-ㅣ가-힝\s]', '', text)     # 정규표현식으로 텍스트 정제
    
    okt = Okt()
    okt_data = okt.pos(text)                       # (단어, 품사) 형태의 튜플 리스트 반환

    words = []
    for word, pos in okt_data:
        if pos in ['Adjective', 'Noun', 'Verb']:   # 형용사, 명사, 동사만 추출
            words.append(word)


    word_str = ' '.join(words)                     # 단어들을 공백으로 연결하여 하나의 문자열로 반환
    return word_str

In [8]:
token_list = []                     # 각 댓글에서 추출된 토큰 문자열을 저장할 리스트

for text in tqdm(df['댓글']):        # 댓글 데이터 처리 진행 상황 표시
    result = tokenize_text(text)    # tokenize_text 함수를 사용해서 텍스트 정제, 단어 추출
    token_list.append(result)

100%|██████████| 460/460 [00:06<00:00, 67.07it/s] 


In [ ]:
token_list

In [ ]:
# 토픽 모델링에서는 너무 짧은 문서가 
corpus_list = []

# 모든 토큰화된 문장을 순회하며 짧은 문장 찾기
for index in range(len(token_list)):
    corpus = token_list[index]

    # 문장을 공백으로 분할 -> set으로 중복 제거 -> 고유 단어 개수 확인
    if len(set(corpus.split())) < 3:        # 고유 단어가 3개 미만인 경우
        corpus_list.append(corpus)

for corpus in corpus_list:
    token_list.remove(corpus)

print(len(token_list))

460


# LDA 작동 과정

1. 모든 단어와 문서를 랜덤하게 주제에 할당
    - 처음에는 랜덤하게 각 단어가 어떤 주제에 속하는지 정한다.
    - 예: "축구"라는 단어가 "스포츠" 주제에 배정될 수도, "정치" 주제에 배정될 수도 있다.

2. 단어의 주제를 반복적으로 재할당
    - 각 단어의 주제 분포를 고려해 더 적합한 주제로 배정한다.
    - 예: "축구"라는 단어가 "스포츠" 주제에서 자주 등장한다면 점점 스포츠 주제로 배정된다.

3. 수렴할 때까지 반복
    - 모든 단어가 가장 적절한 주제에 할당될 때까지 반복한다.

4. 결과 출력: 문서의 토픽 분포
    - 각 문서가 어떤 주제들로 이루어졌는지 확률 값으로 나타낸다.

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [ ]:
count_vectorizer = CountVectorizer(max_df=0.1,              # 단어가 너무 자주(전체 문서의 10%) 등장하면 무시
                                   max_features=1000,       # 최대 피처 수
                                   min_df=2,                # 최소 문서 빈도(2 이상인 경우 포함)
                                   ngram_range=(1,2))       # 1-그램, 2-그램(연속된 두 단어 조합) 모두 사용

In [15]:
# 텍스트 데이터를 피처 벡터로 변환
# 문서-단어 행렬이 만들어짐.
feat_vect = count_vectorizer.fit_transform(token_list) # 각 문서가 어떤 단어를 얼마나 포함하는지 나타내는 벡터

In [16]:
print(feat_vect.shape)              # (문서수, 변환된 단어(피처) 수)
print(count_vectorizer.vocabulary_) # 피처 이름과 인덱스를 포함한 단어 출력

(460, 1000)
{'하려고': np.int64(931), '들어가면': np.int64(201), '최신': np.int64(850), '정보': np.int64(762), '로딩': np.int64(218), '반복': np.int64(304), '이미지': np.int64(660), '나오고': np.int64(116), '상황': np.int64(430), '새로고침': np.int64(432), '심지어': np.int64(503), '지금': np.int64(809), '않음': np.int64(556), '때문': np.int64(207), '서버': np.int64(440), '건지': np.int64(50), '주문 취소': np.int64(794), '취소 하려고': np.int64(858), '하나': np.int64(908), '번호': np.int64(364), '인증': np.int64(677), '카드': np.int64(860), '결제': np.int64(62), '선택': np.int64(443), '정도': np.int64(759), '해야': np.int64(960), '수단': np.int64(460), '아예': np.int64(530), '없고': np.int64(584), '현금': np.int64(980), '옵션': np.int64(624), '시켜': np.int64(481), '먹는데': np.int64(257), '없습니다': np.int64(591), '무슨': np.int64(287), '핸드폰': np.int64(968), '불편하게': np.int64(392), '저녁': np.int64(736), '방법': np.int64(317), '없네요': np.int64(586), '카드 결제': np.int64(861), '결제 수단': np.int64(63), '업데이트': np.int64(577), '하기': np.int64(907), '점점': np.int64(755), '클릭': np.int64(

In [ ]:
# 고유 단어 목록 출력
count_vectorizer.get_feature_names_out()

In [ ]:
# 모델 선언
# 토픽 수가 너무 적으면 서로 다른 이슈가 하나로 뭉칠 수 있음. 반대로 너무 많으면 비슷한 토픽이 여러 개로 쪼개져 해석이 어려워질 수 있음.
lda = LatentDirichletAllocation(n_components=5) # 추출할 토픽 5개

In [25]:
# 모델 학습
lda.fit(feat_vect)


,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",5
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [26]:
feature_names = count_vectorizer.get_feature_names_out()

for topic_index, topic in enumerate(lda.components_):
    print('Topic', topic_index + 1)
    topic_word_indexes = topic.argsort()[::-1]

    # 현재 토픽에서 가중치가 가장 높은 상위 10개 단어 인덱스만 선택
    top_index = topic_word_indexes[0:10]
    print(top_index)

    # 인덱스를 실제 단어로 변환
    feature_concat = [feature_names[i] for i in top_index]
    print(feature_concat)


Topic 1
[557 866  55 558 415 667 653 694 560  30]
['알뜰', '쿠폰', '검색', '알뜰 배달', '사진', '이용', '이런', '있는', '알림', '같은']
Topic 2
[ 62 819 577 447 107  50 816 905 609 331]
['결제', '진짜', '업데이트', '설정', '기사', '건지', '지연', '하고', '예상', '배달 시간']
Topic 3
[421 833 606 359 290 441 948  88 866 213]
['상담', '채팅', '연락', '배차', '문의', '서비스', '할인', '그냥', '쿠폰', '라이더']
Topic 4
[990 216  55 417 734 804 363 715 443 991]
['화면', '로그인', '검색', '삭제', '장바구니', '주소', '버튼', '있으면', '선택', '확인']
Topic 5
[ 62 667 812 493 706 854 443 434 641 416]
['결제', '이용', '지도', '식당', '있습니다', '추가', '선택', '생각', '위치', '사항']


In [27]:
# LDA 모델에서 각 토픽을 대표하는 상위 단어를 출력하는 함수
def display_topic(model, feature_names, num_top_words):
    for topic_index, topic in enumerate(model.components_):
        print('Topic', topic_index + 1)
        topic_word_indexes = topic.argsort()[::-1]
        top_index = topic_word_indexes[0:num_top_words]
        print(top_index)
        feature_concat = [feature_names[i] for i in top_index]
        print(feature_concat)

In [28]:
# 학습된 LDA 모델에서 각 토픽별 상위 10개 단어를 출력
display_topic(lda, feature_names, 10)

Topic 1
[557 866  55 558 415 667 653 694 560  30]
['알뜰', '쿠폰', '검색', '알뜰 배달', '사진', '이용', '이런', '있는', '알림', '같은']
Topic 2
[ 62 819 577 447 107  50 816 905 609 331]
['결제', '진짜', '업데이트', '설정', '기사', '건지', '지연', '하고', '예상', '배달 시간']
Topic 3
[421 833 606 359 290 441 948  88 866 213]
['상담', '채팅', '연락', '배차', '문의', '서비스', '할인', '그냥', '쿠폰', '라이더']
Topic 4
[990 216  55 417 734 804 363 715 443 991]
['화면', '로그인', '검색', '삭제', '장바구니', '주소', '버튼', '있으면', '선택', '확인']
Topic 5
[ 62 667 812 493 706 854 443 434 641 416]
['결제', '이용', '지도', '식당', '있습니다', '추가', '선택', '생각', '위치', '사항']


In [24]:
# LDA를 통해 각 문서(댓글)가 어떤 토픽에 속하는지 확인
# lda.transform(feat_vect) : 문서별 토픽 확률 분포 반환
topic_result = lda.transform(feat_vect)

sent_topic_list = []

# 각 문서에 대해 가장 높은 확률을 가진 토픽 번호와 그 확률을 추출.
for temp in range(len(topic_result)):
    topic = topic_result[temp].argmax()
    sent_topic_list.append([temp, topic, topic_result[temp].max()])


topic_df = pd.DataFrame(sent_topic_list, columns=['no', '토픽번호', '확률'])

topic_df['댓글'] = df['댓글']

topic_df


,no,토픽번호,확률,댓글
0,0,4,0.542394,80분 걸린다길래 주문취소 하려고 주문내역에 들어가면 계속 최신 정보를 불러오지 못...
1,1,2,0.979586,음식 하나 시키는데 우리나라 앱들은 1) 국내 번호 필요함. 번호 인증 필수 2) ...
2,2,1,0.880724,왜이렇게 업데이트 할때마다 사용하기 점점 불편하게 바뀌는지? 클릭한번 더 해야되고 ...
3,3,4,0.974521,"배달의 민족앱자체는 만족하나, 식사후 맛 리뷰 평점 자체는 클린하게 이뤄지진 못하는..."
4,4,1,0.978039,장바구니가 너무 불편합니다. 비마트에서 여러가지를 담고 스크롤 올리고 내릴때 살짝 ...
...,...,...,...,...
455,455,1,0.958940,우선 배달업체 광고가 너무 많습니다. 두번째는 주문직전 주소바꾸기 안되는게 매우매우...
456,456,0,0.565428,배민 쭉 써왔고 쓴소리 하나 하려합니다. 중간다리 플랫폼으로서 식당/유저 사이 중재...
457,457,1,0.956803,업데이트된거 디자인 너무 불편해요. 배민1 부분에서 음식점 둘러보는데 빨리 한거번에...
458,458,2,0.542337,첫주문도 아닌데 첫주문 할인받으로 광고 계속오고 친구초대 하려하니까 주문내역이 없다...


In [ ]:
# 토픽 단어만 보고 이름을 붙이는 것은 위험할 수 있음
topic0 = topic_df[topic_df['토픽번호']==0]
topic0.sort_values(by='확률', ascending=False).head(10)


,no,토픽번호,확률,댓글
236,236,0,0.990328,업데이트에 불만이 있습니다. 기존 버전에선 배민오더로 주문할 시 지도를 이용해 주문...
335,335,0,0.985983,한창 장바구니에 메뉴 고르던중에 영업시간이 종료되는 경우도 있어 아쉬워요. 영업 마...
323,323,0,0.980738,포장 선택시 기존 지도상에서 보이던 가게들이 목록으로만 볼수 있게 봐뀐점은 너무 불...
428,428,0,0.977932,1. 음식점 리스트 중 수수료 가장 비싼 오픈리스트. 기존 랜덤 3개가 최상단에 떴...
24,24,0,0.977367,쿠폰을 상시로 뿌리고 가끔 이벤트로 뿌리는 쿠폰까지 더하면 포장방문 보다 약 10~...
433,433,0,0.974530,이벤트성 페이지에서 오류가 종종 발생하는데 실시간 상담같은 cs로 상세한 내용과 분...
286,286,0,0.972699,장바구니 상하로 플릭킹하는데 자꾸 좌우 탭이동으로 오작동한다는 말을 왜 못알아듣는척...
325,325,0,0.970877,"몇년 전에는 다른 배달 앱 쓰다가 배달의민족만 계속쓰고 있는데요, 아쉬운 점이 있습..."
372,372,0,0.969742,배달의 민족 항상 잘 쓰고 있습니다. 그런데 한 휴대폰에서 두 개 이상의 계정을 사...
91,91,0,0.968219,배달 너무 좋은어플이라 잘 쓰고 있던 유저였어요. 어느날 실수로 다른 주소로 주문을...


In [ ]:
# %pip install pyLDAvis
# uv add pyLDAvis

In [ ]:
# 시각화
import pyLDAvis.lda_model


# Jupyter Notebook 환경에서 시각화 활성화
pyLDAvis.enable_notebook()

vis = pyLDAvis.lda_model.prepare(
    lda,                            # 학습된 LDA 모델
    feat_vect,                      # CountVectorizer로 만든 문서-단어 행렬
    count_vectorizer                # 단어 사전 정보를 가진 vectorizer
)

pyLDAvis.display(vis)

In [ ]:
'''
왼쪽 : 토픽 간 거리 지도
- 원 하나가 하나의 토픽.
- 원의 크기 : 해당 토픽이 전체 문서에서 차지하는 비중
- 원끼리 가까우면 단어 분포가 비슷한 토픽
- 겹쳐 있으면 토픽이 명확히 분리되지 않았다는 뜻.

오른쪽 막대 그래프 : 선택된 토픽의 주요 단어
- 파란색 막대 : 전체 데이터에서 해당 단어가 등장한 빈도
- 빨간색 막대 : 선택된 토픽 안에서 해당 단어가 등장한 추정 빈도

오른쪽 위 λ 슬라이더
- λ = 1.0에 가까우면 해당 토픽 안에서 자주 등장하는 단어 중심으로 정렬
- λ = 0.0에 가까우면 전체 데이터에서는 덜 흔하지만 해당 토픽에서 특히 두드러지는 단어가 강조.
- 보통 0.5~0.6 전후로 조정하면서 보는 경우가 많음.
'''